In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import re
import json
import pickle
from rank_bm25 import BM25Okapi

In [4]:
# Load chunks
with open("../data/processed/chunks/trafik_kanunu.jsonl", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

In [5]:
# Clean the texts
def tokenize_tr(text: str) -> list[str]:
    text = text.replace("İ", "i").replace("I", "ı").lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)

    return text.split()

In [6]:
chunks[0]["text"]

'KARAYOLLARI TRAFİK KANUNU123\nKanun Numarası : 2918\nKabul Tarihi : 13/10/1983\nYayımlandığı Resmî Gazete : Tarih: 18/10/1983 Sayı: 18195\nYayımlandığı Düstur : Tertip: 5 Cilt: 22 Sayfa: 687\nBİRİNCİ KISIM\nGenel Esaslar\nBİRİNCİ BÖLÜM\nAmaç ve Kapsam\nAmaç:'

In [7]:
tokenize_tr(chunks[0]["text"])

['karayolları',
 'trafik',
 'kanunu123',
 'kanun',
 'numarası',
 '2918',
 'kabul',
 'tarihi',
 '13',
 '10',
 '1983',
 'yayımlandığı',
 'resm',
 'gazete',
 'tarih',
 '18',
 '10',
 '1983',
 'sayı',
 '18195',
 'yayımlandığı',
 'düstur',
 'tertip',
 '5',
 'cilt',
 '22',
 'sayfa',
 '687',
 'birinci',
 'kısım',
 'genel',
 'esaslar',
 'birinci',
 'bölüm',
 'amaç',
 've',
 'kapsam',
 'amaç']

In [8]:
tokenized_corpus = [tokenize_tr(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

In [9]:
with open("../data/processed/bm25_index.pkl", "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": chunks}, f)

In [10]:
query = "Yaya kuralları nelerdir?"
top_k = 5

In [11]:
tokenized_query = tokenize_tr(query)
scores = bm25.get_scores(tokenized_query)
scored = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]

results = [(chunks[idx], float(score)) for idx, score in scored]

In [12]:
for chunk, score in results:
    print(f"Madde {chunk['metadata']['madde_no']} | Score: {score:.3f}")
    print(chunk["text"][:150])
    print("---")

Madde 55 | Score: 6.952
52 – Sürücüler:
a) Kavşaklara yaklaşırken, dönemeçlere girerken, tepe üstlerine yaklaşırken, dönemeçli
yollarda ilerlerken, yaya geçitlerine, hemzemin
---
Madde 72 | Score: 4.942
68 – Yayaların uyacakları kurallar aşağıda belirtilmiştir.
58 Anayasa Mahkemesinin 14/3/2019 tarihli ve E.: 2019/1, K.: 2019/14 sayılı Kararı ile, bu 
---
Madde 57 | Score: 4.885
54 – Sürücülerin geçme sırasında uymak zorunda oldukları kural ve yasaklar
şunlardır:
a) Geçme kuralları:
Sürücülerin önlerinde giden bir aracı geçmel
---
Madde 78 | Score: 3.846
74 – (Başlığı ile Birlikte Değişik: 18/10/2018-7148/25 md.)
Sürücüler, görevli bir kişi veya ışıklı trafik işareti bulunmayan ancak trafik işareti vey
---
Madde 48 | Score: 3.809
45 – (Değişik: 12/7/2013-6495/19 md.)
Sürücü belgesi sahibi kişide sağlığı bakımından sürücülüğe engel aşikar bir değişikliğin
görülmesi ve tespiti hâ
---
